# SCRIPT 1 — DATA PIPELINE
## Brahmaputra Morphodynamics Study (AP-GARSS 2026)

**Purpose:**
- Part A: Generate 4-class labelled training tiles from the Training Reach (upper Assam)
- Part B: Export full-scene multimodal imagery for 7 annual inference time steps (Tezpur–Guwahati reach, 2019–2025)

**Class scheme:**
- 0 = Background (outside braidplain or non-river surfaces)
- 1 = Active Channel (water)
- 2 = Active Bar (bare/sparsely vegetated sand-gravel)
- 3 = Vegetated Bar (spatial residual inside braidplain after classes 1 and 2 removed — NO NDVI used)

**Label logic for Class 3 (Vegetated Bar):**  
Any pixel inside the 28-year braidplain mask that is neither Active Channel nor Active Bar  
is by spatial definition a vegetated/stabilised bar surface. This avoids NDVI leakage  
since the model inputs include raw NIR/Red bands from which NDVI could be derived.

**Training reach:** Upper Assam ~94.7–95.3°E (spatially separated from inference reach)
**Inference reach:** Tezpur to Guwahati ~91.5–92.8°E (downstream of Bhomoraguri gauge)
**Gauge:** Bhomoraguri, Sonitpur — upstream forcing signal for inference reach

## 0. Install and Imports

In [ ]:
# !pip install earthengine-api geemap numpy tqdm rasterio -q

import ee
import geemap
import numpy as np
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio

print('Imports complete.')

In [ ]:
ee.Authenticate()
ee.Initialize(project='rahulds001')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration — Paths, Assets, Parameters

In [ ]:
# ── Drive paths ─────────────────────────────────────────────────────────
ROOT          = '/content/drive/MyDrive/BRAHMA_MORPHO'

# Training data output dirs (tiled numpy)
TILES         = f'{ROOT}/training/tiles'
SPLITS        = f'{ROOT}/training/splits'
META          = f'{ROOT}/training/metadata'
LOGS          = f'{ROOT}/logs'
FIGURES       = f'{ROOT}/figures'

# Inference scene output dir (full-scene GeoTIFFs, one per year)
INFERENCE_DIR = f'{ROOT}/inference/scenes'

for d in [
    TILES, META, LOGS, FIGURES,
    f'{SPLITS}/train',
    f'{SPLITS}/val',
    f'{SPLITS}/test',
    INFERENCE_DIR
]:
    os.makedirs(d, exist_ok=True)

print('Directories ready.')

In [ ]:
# ── GEE Asset paths ─────────────────────────────────────────────────────
# These are your existing validated assets — do not change
BRAIDPLAIN_ASSET = 'projects/rahulds001/assets/braidplain_mask_v1'   # 28-yr Landsat MNDWI, manually corrected
CORRIDOR_ASSET   = 'projects/rahulds001/assets/AMRD_corridor_final'
WMAX_ASSET       = 'projects/rahulds001/assets/AMRD_WMax_1988_2025'

print('Assets defined.')

In [ ]:
# ── Sensor and tile parameters ──────────────────────────────────────────
RESOLUTION    = 10           # metres — Sentinel-2 native optical resolution
TILE_SIZE_PX  = 256          # pixels per training tile side
TILE_SIZE_M   = TILE_SIZE_PX * RESOLUTION   # = 2560 m
OVERLAP_M     = 256          # tile overlap in metres
CRS           = 'EPSG:32646' # UTM Zone 46N — consistent with your existing pipeline
MAX_CLOUD     = 10           # % cloud cover threshold for Sentinel-2
S1_PASS       = 'DESCENDING' # Sentinel-1 orbit pass — consistent with existing pipeline

# ── Input band names (9 channels total — same as existing pipeline) ──────
S2_BANDS      = ['B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']  # 7 optical bands
S1_BANDS      = ['VV', 'VH']                                      # 2 SAR bands
# Note: NDVI and NDWI are NOT included as model inputs to avoid leakage.
#       They are used only for label generation and are never saved as input channels.

print(f'Parameters set. Tile: {TILE_SIZE_PX}px = {TILE_SIZE_M}m. CRS: {CRS}.')
print(f'Input channels: {S2_BANDS + S1_BANDS}  ({len(S2_BANDS)+len(S1_BANDS)} total)')

In [ ]:
# ── Study area geometries ────────────────────────────────────────────────

# TRAINING REACH — Upper Assam (model learning domain)
# Spatially separated ~180km from inference reach
# TRAINING_AOI = ee.Geometry.Rectangle([94.5, 27.1, 95.3, 27.7])
TRAINING_AOI = ee.FeatureCollection('projects/rahulds001/assets/Train_AOI').geometry()
# INFERENCE REACH — Tezpur to Guwahati (monitoring domain)
# Downstream of Bhomoraguri gauge station (~92.7°E)
# This reach is NEVER used for training — pure inference only

INFERENCE_AOI = ee.Geometry.Rectangle(
    [91.70033447265615,   # xmin (west)
     26.080002725994902,  # ymin (south)
     92.8566455078124,    # xmax (east)
     26.72086332207767],  # ymax (north)
    None,
    False
)

# ── Annual lean season time windows for inference (2019-2025) ────────────
# February–March of each year: cloud-free, minimum discharge, bars fully exposed
# NOTE: 2022-2023 may have reduced S1 coverage due to Sentinel-1B retirement (Aug 2022)
#       Check coverage in GEE before committing — use wider date window if needed
INFERENCE_YEARS = {
    '2019': ('2019-02-01', '2019-03-31'),
    '2020': ('2020-02-01', '2020-03-31'),
    '2021': ('2021-02-01', '2021-03-31'),
    '2022': ('2022-02-01', '2022-03-31'),  # check S1 coverage
    '2023': ('2023-02-01', '2023-03-31'),  # check S1 coverage
    '2024': ('2024-02-01', '2024-03-31'),
    '2025': ('2025-02-01', '2025-03-31'),
}

# Training time window — single lean season composite for label generation
TRAIN_START = '2024-01-15'
TRAIN_END   = '2024-03-15'

TRAIN_START_2 = '2025-01-15'
TRAIN_END_2   = '2025-03-15'

# ── Confirmed cluster assignments — training reach ───────────────────────
# Set after visual inspection via visualise_clusters()
# These are the DEFAULT values used when build_label() is called
# without explicit cluster parameters (i.e. during training extraction)
# DO NOT change these — they are locked to your training reach inspection
ACTIVE_BAR_CLUSTERS    = [1, 2, 3]   # bare sand, moist sand, bright dry sand
VEGETATED_BAR_CLUSTERS = [0, 4]      # sparse pioneer veg, dense bar vegetation

# Inference reach clusters — set AFTER running visualise_clusters_inference()
# Leave as None until confirmed visually
INFERENCE_ACTIVE_BAR_CLUSTERS    = [2,3,4]   # ← fill after visual check
INFERENCE_VEGETATED_BAR_CLUSTERS = [0,1]   # ← fill after visual check

print(f'Training reach — Active Bar    : {ACTIVE_BAR_CLUSTERS}')
print(f'Training reach — Vegetated Bar : {VEGETATED_BAR_CLUSTERS}')
print('Inference reach clusters: set after visualise_clusters_inference()')

print('AOIs and time windows defined.')
print(f'Training reach: {TRAINING_AOI.getInfo()["coordinates"]}')
print(f'Inference reach: {INFERENCE_AOI.getInfo()["coordinates"]}')
print(f'Inference years: {list(INFERENCE_YEARS.keys())}')

## 2. Utility Functions

In [ ]:
# ── Otsu threshold (preserved exactly from your existing pipeline) ────────

def otsu(hist_result):
    counts = ee.Array(ee.Dictionary(hist_result).get('histogram'))
    means  = ee.Array(ee.Dictionary(hist_result).get('bucketMeans'))
    size   = means.length().get([0])
    total  = counts.reduce(ee.Reducer.sum(),  [0]).get([0])
    sum_   = counts.multiply(means).reduce(ee.Reducer.sum(), [0]).get([0])
    mean   = sum_.divide(total)
    bss    = ee.List.sequence(1, size).map(lambda i: (
        lambda aC, aM, bC, bMean:
            aC.multiply(aM.subtract(mean).pow(2))
             .add(bC.multiply(bMean.subtract(mean).pow(2)))
    )(
        ee.Array(counts.slice(0, 0, i)).reduce(ee.Reducer.sum(), [0]).get([0]),
        ee.Array(counts.slice(0, 0, i)).multiply(
            ee.Array(means.slice(0, 0, i))
        ).reduce(ee.Reducer.sum(), [0]).get([0]).divide(
            ee.Array(counts.slice(0, 0, i)).reduce(ee.Reducer.sum(), [0]).get([0])
        ),
        total.subtract(
            ee.Array(counts.slice(0, 0, i)).reduce(ee.Reducer.sum(), [0]).get([0])
        ),
        sum_.subtract(
            ee.Array(counts.slice(0, 0, i)).reduce(ee.Reducer.sum(), [0]).get([0])
            .multiply(
                ee.Array(counts.slice(0, 0, i)).multiply(
                    ee.Array(means.slice(0, 0, i))
                ).reduce(ee.Reducer.sum(), [0]).get([0]).divide(
                    ee.Array(counts.slice(0, 0, i)).reduce(ee.Reducer.sum(), [0]).get([0])
                )
            )
        ).divide(
            total.subtract(
                ee.Array(counts.slice(0, 0, i)).reduce(ee.Reducer.sum(), [0]).get([0])
            )
        )
    ))
    return means.sort(bss).get([-1])


def safe_otsu(band_image, band_name, aoi, fallback):
    """Otsu threshold with safe fallback if histogram is degenerate."""
    hist = band_image.reduceRegion(
        reducer   = ee.Reducer.histogram(minBucketWidth=0.01),
        geometry  = aoi,
        scale     = 30,
        bestEffort= True,
        maxPixels = 1e10
    )
    hist_obj  = ee.Dictionary(hist)
    has_band  = hist_obj.contains(band_name)
    threshold = ee.Algorithms.If(
        has_band,
        ee.Algorithms.If(
            ee.List(
                ee.Dictionary(
                    hist_obj.get(band_name,
                        ee.Dictionary({'bucketMeans': ee.List([])}))
                ).get('bucketMeans', ee.List([]))
            ).size().gt(2),
            otsu(hist_obj.get(band_name)),
            ee.Number(fallback)
        ),
        ee.Number(fallback)
    )
    return ee.Number(threshold)

print('Otsu functions defined.')

In [ ]:
# ── Cloud masking function for Sentinel-2 ───────────────────────────────
def mask_s2_clouds(image):
    """
    Mask clouds and cirrus using Sentinel-2 SCL band (Scene Classification Layer).
    SCL values kept: 4=vegetation, 5=bare soil, 6=water, 7=unclassified
    SCL values masked: 1=saturated, 2=dark pixels, 3=cloud shadow,
                       8=cloud medium prob, 9=cloud high prob, 10=cirrus, 11=snow
    """
    scl     = image.select('SCL')
    # Keep only clear land, water, and unclassified pixels
    clear   = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    return image.updateMask(clear)


# ── TEMPORALLY JOINED S1-S2 COMPOSITE BUILDER ────────────────────────────
# Uses saveBest join to pair each S2 scene with its closest S1 acquisition
# within a 2-day window. Guarantees all output bands have valid data.
# This replaces the independent mosaic approach which caused blank SAR bands.

def build_s1s2_composite_joined(aoi, start, end,
                                 cloud_threshold=10,
                                 orbit_pass=S1_PASS):
    """
    Build temporally joined S1-S2 composite.
    Only keeps S2 scenes that have a matching S1 acquisition within 2 days.
    Returns median composite of all valid joined pairs.

    Parameters
    ----------
    aoi              : ee.Geometry
    start            : str — 'YYYY-MM-DD'
    end              : str — 'YYYY-MM-DD'
    cloud_threshold  : int — max cloud % for S2 (default 10, relax if needed)
    orbit_pass       : str — 'DESCENDING' or 'ASCENDING'

    Returns
    -------
    stacked  : ee.Image — 9-band median composite (S2_BANDS + S1_BANDS)
    s2       : ee.Image — S2 bands only
    s1       : ee.Image — S1 bands only
    n_pairs  : int     — number of valid S1-S2 pairs found
    """

    # ── Sentinel-2 — cloud filtered ──────────────────────────────────
    s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(aoi)
              .filterDate(start, end)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
              .select(S2_BANDS + ['SCL'])
              .map(mask_s2_clouds))

    # ── Sentinel-1 — IW mode, specified orbit pass ───────────────────
    s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterBounds(aoi)
              .filterDate(start, end)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.eq('orbitProperties_pass', orbit_pass))
              .filter(ee.Filter.listContains(
                  'transmitterReceiverPolarisation', 'VV'))
              .filter(ee.Filter.listContains(
                  'transmitterReceiverPolarisation', 'VH'))
              .select(S1_BANDS))

    # ── Temporal join — S2 primary, S1 secondary ─────────────────────
    # For each S2 scene find the closest S1 acquisition within 2 days
    # saveBest keeps only the single best match per S2 scene
    max_diff_ms = 2 * 24 * 60 * 60 * 1000   # 2 days in milliseconds

    time_filter = ee.Filter.maxDifference(
        difference  = max_diff_ms,
        leftField   = 'system:time_start',
        rightField  = 'system:time_start'
    )

    join = ee.Join.saveBest(
        matchKey   = 'matched',
        measureKey = 'time_difference'
    )

    joined = join.apply(
        primary   = s2_col,
        secondary = s1_col,
        condition = time_filter
    )

    # ── Merge S1+S2 bands for each joined pair ───────────────────────
    def merge_bands(img):
        s2_img = ee.Image(img).select(S2_BANDS)
        s1_img = ee.Image(img.get('matched')).select(S1_BANDS)
        return s2_img.addBands(s1_img).rename(S2_BANDS + S1_BANDS)

    merged_col = ee.ImageCollection(joined.map(merge_bands))

    # ── Count valid pairs ────────────────────────────────────────────
    n_pairs = merged_col.size().getInfo()

    # ── Median composite of all valid pairs ──────────────────────────
    composite = merged_col.select(S2_BANDS + S1_BANDS).median().clip(aoi)

    s2_out = composite.select(S2_BANDS)
    s1_out = composite.select(S1_BANDS)

    return composite, s2_out, s1_out, n_pairs


print('Temporally joined S1-S2 composite builder defined.')
print('Guarantees all output bands have valid data via 2-day S1-S2 pairing.')

## 3. Four-Class Label Generator (Training Reach)

In [ ]:
# ── REDESIGNED LABEL BUILDER — Index-Based with GMM Refinement ───────────
# Physical basis:
#   Class 0 — Background    : outside braidplain mask
#   Class 1 — Active Channel: MNDWI/VV Otsu (unchanged — robust)
#   Class 2 — Active Bar    : high BSI + low NDVI inside braidplain
#   Class 3 — Vegetated Bar : positive MSAVI2 + low BSI inside braidplain
#                             captures brown/sparse/dried vegetation
#
# Key improvement over raw band clustering:
#   BSI directly measures bare soil/sand exposure
#   MSAVI2 directly measures sparse and stressed vegetation
#   No arbitrary cluster assignment needed
#   Brown/dried vegetation correctly captured by MSAVI2 lower threshold
#   Physically interpretable — each threshold has geomorphological meaning

def build_label(aoi, start, end,
                active_bar_clusters    = None,   # kept for API compatibility
                vegetated_bar_clusters = None):  # kept for API compatibility
    """
    Generate 4-class weak label mask using physically meaningful spectral indices.

    Index-based classification inside braidplain:
    - Active Bar    : BSI > threshold AND NDVI < 0.15
    - Vegetated Bar : MSAVI2 > 0.05 AND BSI < bar_threshold (captures brown/sparse veg)
    - Ambiguous     : remaining braidplain pixels assigned by nearest index neighbour

    Parameters
    ----------
    aoi   : ee.Geometry
    start : str — 'YYYY-MM-DD'
    end   : str — 'YYYY-MM-DD'

    Returns dict: stacked, optical, sar, mask, class1, class2, class3
    """

    # ── Load braidplain mask ──────────────────────────────────────────────
    braidplain = ee.Image(BRAIDPLAIN_ASSET).select('braidplain').eq(1)

    # ── Build temporally joined S1-S2 composite ───────────────────────────
    stacked, s2, s1, n_pairs = build_s1s2_composite_joined(aoi, start, end)
    print(f'  Valid S1-S2 pairs: {n_pairs}')

    # ── Spectral indices — label generation only, not model inputs ─────────

    # MNDWI — water detection (for channel class)
    mndwi = s2.normalizedDifference(['B3', 'B11']).rename('MNDWI')

    # NDWI — supplementary water index
    ndwi  = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')

    # NDVI — standard vegetation index
    ndvi  = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

    # BSI — Bare Soil Index
    # High for exposed mineral surfaces (sand, gravel, bare soil)
    # Low for vegetation and water
    # Range roughly -1 to +1, bare surfaces typically 0.1 to 0.5
    bsi = (s2.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {
            'SWIR': s2.select('B11'),
            'RED' : s2.select('B4'),
            'NIR' : s2.select('B8'),
            'BLUE': s2.select('B2')
        }
    )).rename('BSI')

    # MSAVI2 — Modified Soil Adjusted Vegetation Index 2
    # More sensitive to sparse and stressed vegetation than NDVI
    # Reduces soil background noise — ideal for detecting brown/pioneer bar veg
    # Formula: (2*NIR + 1 - sqrt((2*NIR+1)^2 - 8*(NIR-RED))) / 2
    msavi2 = (s2.expression(
        '(2.0 * NIR + 1.0 - sqrt((2.0 * NIR + 1.0)**2 - 8.0 * (NIR - RED))) / 2.0',
        {
            'NIR': s2.select('B8').divide(10000),   # normalise to 0-1 for formula
            'RED': s2.select('B4').divide(10000)
        }
    )).rename('MSAVI2')

    # SAR VV for supplementary water detection
    vv = s1.select('VV')

    # ── CLASS 1: Active Channel ───────────────────────────────────────────
    # Combined MNDWI/VV Otsu — unchanged from previous version, robust
    mndwi_thresh = safe_otsu(mndwi, 'MNDWI', aoi, -0.1)
    vv_thresh    = safe_otsu(vv,    'VV',    aoi, -16.0)

    s2_water  = mndwi.gt(mndwi_thresh)
    sar_water = vv.lt(vv_thresh)
    high_conf = s2_water.And(sar_water)
    clear_s2  = mndwi.gt(mndwi_thresh.add(0.05))
    clear_sar = vv.lt(vv_thresh.subtract(2))
    either    = clear_s2.Or(clear_sar)

    class1_raw = high_conf.Or(either)
    def_dry    = mndwi.lt(-0.15)

    class1 = (class1_raw
              .where(def_dry, 0)
              .focal_max(radius=60, kernelType='circle', units='meters')
              .focal_min(radius=60, kernelType='circle', units='meters')
              .And(braidplain)              # constrain channel to braidplain
              .rename('active_channel'))

    # ── Residual mask — inside braidplain, not channel ────────────────────
    residual = braidplain.And(class1.Not())

    # ── Compute indices ───────────────────────────────────────────────────────
# Keep all existing index computations — mndwi, ndwi, ndvi, bsi, msavi2

    # ── REMOVE Otsu BSI threshold — replace with fixed physically justified values
    # Otsu fails here because BSI histogram inside braidplain is often unimodal
    # (dominated by sand) with a long vegetated tail — Otsu finds the wrong break

    # ── CLASS 2: Active Bar — independent definition ──────────────────────────
    # Bare sand and gravel: HIGH BSI, LOW NDVI, LOW MSAVI2
    # Use percentile-based threshold — top 40% BSI values inside braidplain
    # are almost certainly bare sand regardless of absolute value
    # This is more robust than Otsu for unimodal BSI distributions

    bsi_p60 = bsi.updateMask(residual).reduceRegion(
        reducer   = ee.Reducer.percentile([60]),
        geometry  = aoi,
        scale     = 30,
        bestEffort= True,
        maxPixels = 1e10
    ).get('BSI') # Corrected key from BSI_p60
    bsi_p60 = ee.Number(bsi_p60)

    bare_bar_raw = (
        bsi.gt(bsi_p60)              # top 40% BSI = most exposed sand surfaces
        .And(ndvi.lt(0.09))          # minimal vegetation signal 0.10
        .And(msavi2.lt(0.09))        # low vegetation even by sensitive index 0.12
        .And(mndwi.lt(0.0))          # not water
        .And(residual)
    )

    class2 = (bare_bar_raw
              .focal_min(radius=30, kernelType='circle', units='meters')
              .focal_max(radius=30, kernelType='circle', units='meters')
              .focal_max(radius=20, kernelType='circle', units='meters')
              .focal_min(radius=20, kernelType='circle', units='meters')
              .And(residual)
              .rename('active_bar'))

    # ── CLASS 3: Vegetated Bar — independent definition ───────────────────────
    # ANY vegetation signal inside braidplain that is not water or bare bar
    # Three sub-populations all captured:
    #
    # Sub-population A — Dense established vegetation (bright red in NIR FC)
    #   NDVI > 0.20, MSAVI2 > 0.15
    #
    # Sub-population B — Sparse/pioneer vegetation (pale pink in NIR FC)
    #   NDVI 0.08-0.20, MSAVI2 0.05-0.15
    #
    # Sub-population C — Brown/dried/stressed vegetation (brown in NIR FC)
    #   NDVI -0.05 to 0.10, MSAVI2 > 0.02, BSI moderate
    #   This is the population that was missing before
    #
    # Union of all three sub-populations = complete vegetated bar class

    # Dense established veg
    veg_dense = (ndvi.gt(0.20)
                .And(msavi2.gt(0.15))
                .And(mndwi.lt(0.0))
                .And(residual)
                .And(class2.Not()))

    # Sparse pioneer veg
    veg_sparse = (ndvi.gt(0.08)
                  .And(ndvi.lte(0.20))
                  .And(msavi2.gt(0.06))
                  .And(mndwi.lt(-0.05))
                  .And(residual)
                  .And(class2.Not()))

    # Brown/dried/stressed veg — the missing population
    # Key insight: brown veg has moderate MSAVI2 (more sensitive than NDVI)
    # but NDVI can be near zero or slightly negative
    # Distinguish from bare sand by: lower BSI than p60 threshold
    #                                 and not spectrally water-like
    # REPLACE veg_brown with tighter definition:
    veg_brown = (msavi2.gt(0.04)               # raised from 0.02 — reduces noise
                .And(ndvi.gt(-0.03))           # raised from -0.08 — excludes shadow
                .And(bsi.lt(bsi_p60))
                .And(bsi.gt(-0.1))             # ADD: exclude very dark surfaces (shadow/water)
                .And(mndwi.lt(-0.10))          # tightened from -0.05 — stronger water guard
                .And(s2.select('B4').gt(500))  # ADD: minimum red reflectance
                                                # wet dark sand has very low B4
                                                # real brown vegetation has moderate B4
                .And(residual)
                .And(class2.Not()))
    # Union of all three sub-populations
    veg_bar_raw = veg_dense.Or(veg_sparse).Or(veg_brown)


    class3 = (veg_bar_raw
              .focal_min(radius=20, kernelType='circle', units='meters')
              .focal_max(radius=20, kernelType='circle', units='meters')
              .focal_max(radius=15, kernelType='circle', units='meters')
              .focal_min(radius=15, kernelType='circle', units='meters')
              .And(residual)
              .And(class2.Not())
              .rename('vegetated_bar'))


    # After class3 morphological cleaning — add connected component filter
# Removes isolated vegetated bar pixels that are almost certainly noise
# Minimum patch size of 900m² = 9 pixels at 10m resolution

    class3_filtered = (class3
                      .connectedPixelCount(maxSize=25, eightConnected=True)
                      .gte(4)              # minimum 4 connected pixels = 400m²
                      .And(class3)
                      .rename('vegetated_bar'))




    # ── Ambiguous residual — assign by BSI proximity ──────────────────────────
    # Pixels inside braidplain not captured by any class
    # Prevents gaps in label mask
    ambiguous = residual.And(class1.Not()).And(class2.Not()).And(class3.Not())

    # High BSI ambiguous → active bar
    # Low BSI ambiguous → vegetated bar
    ambiguous_bar = ambiguous.And(bsi.gt(bsi_p60.multiply(0.6)))
    ambiguous_veg = ambiguous.And(bsi.lte(bsi_p60.multiply(0.6)))

    # ── Assemble final mask ───────────────────────────────────────────────────
    mask = (ee.Image(0)
            .where(braidplain,     3)    # braidplain default = veg bar
            .where(ambiguous_veg,  3)    # ambiguous low BSI → veg bar
            .where(ambiguous_bar,  2)    # ambiguous high BSI → active bar
            .where(class3,         3)    # confirmed vegetated bar
            .where(class2,         2)    # confirmed active bar
            .where(class1,         1)    # active channel — highest priority
            .clip(aoi)
            .rename('mask')
            .toByte())

    print('Index-based labels generated.')
    print('Active Bar    : BSI > p60 percentile + NDVI < 0.10 + MSAVI2 < 0.12')
    print('Vegetated Bar : Three sub-populations:')
    print('  Dense veg   : NDVI > 0.20 + MSAVI2 > 0.15')
    print('  Sparse veg  : NDVI 0.05-0.20 + MSAVI2 > 0.04')
    print('  Brown/dried : MSAVI2 > 0.02 + BSI < p60 + MNDWI < -0.05')

    return {
        'stacked' : stacked,
        'optical' : s2,
        'sar'     : s1,
        'mask'    : mask,
        'class1'  : class1,
        'class2'  : class2,
        'class3'  : class3,
        'bsi'     : bsi,
        'msavi2'  : msavi2,
        'ndvi'    : ndvi,
    }

print('Index-based label builder defined.')
print('BSI for active bar | MSAVI2 for vegetated bar (incl. brown/sparse)')
print('No cluster assignment needed — physically interpretable thresholds')


In [ ]:
def visualise_index_labels(aoi, start, end, zoom=12, center=None):
    """
    Visualise spectral indices and resulting label mask side by side.
    Run before full extraction to verify index-based label quality.
    """
    if center is None:
        center = [27.4, 94.9]   # training reach centre

    print('Building index-based labels...')
    layers = build_label(aoi, start, end)

    Map = geemap.Map(center=center, zoom=zoom)

    # ── Reference imagery ─────────────────────────────────────────────────
    Map.addLayer(
        layers['optical'].clip(aoi),
        {'bands':['B4','B3','B2'], 'min':200, 'max':2500},
        'True Colour', True
    )
    Map.addLayer(
        layers['optical'].clip(aoi),
        {'bands':['B8','B4','B3'], 'min':200, 'max':3500},
        'NIR False Colour', False
    )

    # ── Spectral indices ──────────────────────────────────────────────────
    Map.addLayer(
        layers['bsi'].clip(aoi),
        {'min':-0.3, 'max':0.5,
         'palette':['blue','white','brown']},
        'BSI (brown=high=bare sand)', False
    )
    Map.addLayer(
        layers['msavi2'].clip(aoi),
        {'min':-0.1, 'max':0.6,
         'palette':['brown','yellow','green']},
        'MSAVI2 (green=vegetation incl. sparse)', False
    )
    Map.addLayer(
        layers['ndvi'].clip(aoi),
        {'min':-0.2, 'max':0.6,
         'palette':['brown','yellow','darkgreen']},
        'NDVI', False
    )

    # ── Individual classes ────────────────────────────────────────────────
    Map.addLayer(
        layers['class1'].selfMask().clip(aoi),
        {'palette':['#2196F3']},
        'Class 1 — Active Channel (blue)', False
    )
    Map.addLayer(
        layers['class2'].selfMask().clip(aoi),
        {'palette':['#BDBDBD']},
        'Class 2 — Active Bar (grey)', False
    )
    Map.addLayer(
        layers['class3'].selfMask().clip(aoi),
        {'palette':['#4CAF50']},
        'Class 3 — Vegetated Bar (green)', False
    )

    # ── Full label mask ───────────────────────────────────────────────────
    Map.addLayer(
        layers['mask'].clip(aoi),
        {
            'min'    : 0,
            'max'    : 3,
            'palette': ['#212121','#2196F3','#BDBDBD','#4CAF50']
        },
        'Full Label Mask', True
    )

    # ── Braidplain boundary ───────────────────────────────────────────────
    braidplain = ee.Image(BRAIDPLAIN_ASSET).select('braidplain').eq(1)
    Map.addLayer(
        braidplain.selfMask().clip(aoi),
        {'palette':['#FF6F00'], 'opacity':0.25},
        'Braidplain Mask', False
    )

    print('\nQA Instructions:')
    print('1. Toggle BSI — brown areas should match bare sand bars in NIR FC')
    print('2. Toggle MSAVI2 — green areas should match ALL vegetation on bars')
    print('   including brown/sparse/dried vegetation')
    print('3. Toggle Class 2 — should capture all bare sand surfaces')
    print('4. Toggle Class 3 — should capture full range of bar vegetation')
    print('   including brown tones not captured by NDVI alone')
    print('5. Toggle Full Mask — check for gaps or misassignments')
    print('6. If satisfied — run full extraction')
    print('7. If not — adjust MSAVI2 threshold (currently 0.05)')
    print('   or NDVI upper bound for active bar (currently 0.15)')

    return Map, layers

# QA on training reach first
Map, layers = visualise_index_labels(
    TRAINING_AOI, TRAIN_START_2, TRAIN_END_2,
    zoom   = 12,
    center = [27.4, 94.9]
)
Map

## 4. Training Tile Grid and Extraction Functions

In [ ]:
# ── Tile grid generator ─────────────────────────────────────────────────

def generate_tile_grid(aoi, tile_size_m, overlap_m, crs):
    """
    Generate non-overlapping (with optional overlap) tile bounding boxes over AOI.
    Returns list of ee.Geometry.Rectangle objects.
    """
    bounds = aoi.bounds(1, crs).coordinates().get(0).getInfo()
    xs = [c[0] for c in bounds]
    ys = [c[1] for c in bounds]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    step = tile_size_m - overlap_m
    tiles = []
    x = x_min
    while x < x_max:
        y = y_min
        while y < y_max:
            tile = ee.Geometry.Rectangle(
                [x, y, x + tile_size_m, y + tile_size_m],
                proj=crs, geodesic=False
            )
            tiles.append(tile)
            y += step
        x += step
    print(f'  Generated {len(tiles)} candidate tiles')
    return tiles

print('Tile grid generator defined.')

In [ ]:
# ── Single tile extraction to numpy ─────────────────────────────────────

def extract_tile(layers, tile_geom, tile_id, site_name, force_reextract=False):
    tile_dir = os.path.join(TILES, site_name, tile_id)
    os.makedirs(tile_dir, exist_ok=True)

    # Skip if already extracted UNLESS force reextract requested
    if (not force_reextract and
            os.path.exists(os.path.join(tile_dir, 'mask.npy'))):
        return True, 'skipped'

    # ... rest of function unchanged

    try:
        # ── Extract 9-channel stacked input image ────────────────
        image_np = geemap.ee_to_numpy(
            layers['stacked'],
            bands  = S2_BANDS + S1_BANDS,
            region = tile_geom,
            scale  = RESOLUTION
        )  # shape: (H, W, 9)

        # ── Extract 4-class label mask ───────────────────────────
        mask_np = geemap.ee_to_numpy(
            layers['mask'],
            bands  = ['mask'],
            region = tile_geom,
            scale  = RESOLUTION
        )  # shape: (H, W, 1)

        # ── Crop to exact tile size ──────────────────────────────
        # GEE may return slightly more pixels due to reprojection
        def crop(arr):
            if arr is None:
                return None
            return arr[:TILE_SIZE_PX, :TILE_SIZE_PX, :]

        image_np = crop(image_np)
        mask_np  = crop(mask_np)

        # ── Validate shapes ──────────────────────────────────────
        expected = (TILE_SIZE_PX, TILE_SIZE_PX)
        for name, arr in [('image', image_np), ('mask', mask_np)]:
            if arr is None:
                return False, f'{name} returned None'
            if arr.shape[:2] != expected:
                return False, f'{name} shape {arr.shape} != expected {expected}'

                    # ── Inside extract_tile() — REPLACE the old braidplain filter ───────
            mask_vals     = mask_np[:, :, 0]
            total_px      = mask_vals.size
            braidplain_px = int(np.sum(mask_vals > 0))
            background_px = int(np.sum(mask_vals == 0))

            # Reject ONLY if tile is entirely nodata — no real pixels at all
            # Both braidplain-dominant AND background-dominant tiles are valid
            if braidplain_px == 0 and background_px == 0:
                return False, 'tile_empty_nodata'

            # Tag tile type for stratified sampling in create_splits()
            tile_type = 'braidplain' if braidplain_px / total_px >= 0.10 else 'background'



        # ── Save arrays ──────────────────────────────────────────
        np.save(os.path.join(tile_dir, 'image.npy'), image_np.astype(np.float32))
        np.save(os.path.join(tile_dir, 'mask.npy'),  mask_np.astype(np.uint8))

        # ── Save metadata JSON ───────────────────────────────────
        bounds = tile_geom.bounds(1, CRS).coordinates().get(0).getInfo()
        xs_ = [c[0] for c in bounds]
        ys_ = [c[1] for c in bounds]

        class_labels = {
            0: 'background',
            1: 'active_channel',
            2: 'active_bar',
            3: 'vegetated_bar'
        }
        class_counts = {class_labels[c]: int(np.sum(mask_vals == c))
                        for c in range(4)}
        class_fractions = {k: round(v / total_px, 4)
                           for k, v in class_counts.items()}

        meta = {
            'tile_id'          : tile_id,
            'tile_type'         : tile_type,
            'site'             : site_name,
            'train_start'      : TRAIN_START,
            'train_end'        : TRAIN_END,
            'crs'              : CRS,
            'resolution_m'     : RESOLUTION,
            'tile_size_px'     : TILE_SIZE_PX,
            'x_min'            : min(xs_),
            'x_max'            : max(xs_),
            'y_min'            : min(ys_),
            'y_max'            : max(ys_),
            'class_counts'     : class_counts,
            'class_fractions'  : class_fractions,
            'channels'         : {
                'image': S2_BANDS + S1_BANDS,    # 9 channels
                'mask' : ['class_0_to_3']         # 4 classes
            },
            'class_definitions': {
                '0': 'background — outside braidplain or non-river surface',
                '1': 'active_channel — MNDWI/VV water detection',
                '2': 'active_bar — high-reflectance bare sand/gravel inside braidplain',
                '3': 'vegetated_bar — spatial residual inside braidplain (no NDVI)'
            }
        }
        with open(os.path.join(META, f'{tile_id}.json'), 'w') as f:
            json.dump(meta, f, indent=2)

        return True, 'extracted'

    except Exception as e:
        return False, str(e)

print('Tile extraction function defined.')
print('Output: image.npy (9ch float32) + mask.npy (4-class uint8)')

In [ ]:
import shutil

# Clear only the QA cache — not any real training tiles
qa_cache = os.path.join(TILES, 'qa')
if os.path.exists(qa_cache):
    shutil.rmtree(qa_cache)
    print('QA cache cleared')



## 5. PART A — Run Training Tile Extraction

In [ ]:
# ── QA check: visualise 10 random sample tiles ───────────────────────────
# Randomly samples 10 braidplain-intersecting tiles and displays them.
# Run before full extraction to verify label quality.
# Re-run multiple times — different random tiles each time.

def qa_sample_tiles(n_samples=10):
    import random

    print('Building labels for training reach...')
    layers = build_label(TRAINING_AOI, TRAIN_START, TRAIN_END)

    print('Generating tile grid...')
    tiles = generate_tile_grid(TRAINING_AOI, TILE_SIZE_M, OVERLAP_M, CRS)

    # ── Find all tiles that intersect braidplain ─────────────────────
    print('Scanning for braidplain-intersecting tiles...')
    braidplain_img  = ee.Image(BRAIDPLAIN_ASSET).select('braidplain').eq(1)
    valid_tiles     = []

    for idx, tile_geom in enumerate(tqdm(tiles, desc='Scanning tiles')):
        bp_sum = braidplain_img.reduceRegion(
            reducer=ee.Reducer.sum(), geometry=tile_geom,
            scale=100, maxPixels=1e6
        ).getInfo()
        val = list(bp_sum.values())[0] if bp_sum else 0
        if val and val > 50:
            valid_tiles.append((idx, tile_geom))

    print(f'  Found {len(valid_tiles)} braidplain-intersecting tiles')

    if len(valid_tiles) == 0:
        print('ERROR: No tiles intersect braidplain — check AOI geometry')
        return

    # ── Randomly sample n_samples tiles ─────────────────────────────
    random.seed()   # no fixed seed — different sample every run
    sampled = random.sample(valid_tiles, min(n_samples, len(valid_tiles)))
    print(f'  Randomly selected {len(sampled)} tiles for QA visualisation')

    # ── Extract each sampled tile ────────────────────────────────────
    qa_data   = []
    names     = {0:'Background', 1:'Active Channel',
                 2:'Active Bar',  3:'Vegetated Bar'}
    cmap      = mcolors.ListedColormap(['#212121', '#2196F3', '#BDBDBD', '#4CAF50'])
    norm      = mcolors.BoundaryNorm([0, 1, 2, 3, 4], cmap.N)

    for tile_idx, tile_geom in tqdm(sampled, desc='Extracting QA tiles'):
        tile_id  = f'QA_tile_{tile_idx:04d}'
        # Inside qa_sample_tiles() — change this line:
        success, status = extract_tile(
            layers, tile_geom, tile_id, 'qa',
            force_reextract=True    # ← always reextract during QA
        )
        if not success:
            print(f'  Skipping tile {tile_idx}: {status}')
            continue

        tile_dir  = os.path.join(TILES, 'qa', tile_id)
        image_np  = np.load(os.path.join(tile_dir, 'image.npy'))  # (256,256,9)
        mask_np   = np.load(os.path.join(tile_dir, 'mask.npy'))   # (256,256,1)
        qa_data.append((tile_idx, image_np, mask_np[:, :, 0]))

    if not qa_data:
        print('No tiles extracted successfully.')
        return

    # ── Visualise — 4 columns per tile: RGB, NIR-FC, SAR-VV, Mask ───
    n_rows = len(qa_data)
    n_cols = 4
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 4, n_rows * 4)
    )

    # Ensure axes is always 2D even if only 1 row
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['True Colour\n(B4,B3,B2)',
                  'NIR False Colour\n(B8,B4,B3)',
                  'SAR VV',
                  '4-Class Label\n0=BG 1=Channel 2=Bar 3=VegBar']

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=9, fontweight='bold')

    for row, (tile_idx, image_np, mask_vals) in enumerate(qa_data):

        # ── Col 0: True Colour RGB ────────────────────────────────
        rgb = image_np[:, :, [2, 1, 0]].astype(np.float32)
        rgb = np.clip((rgb - 200) / (2500 - 200), 0, 1)
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_ylabel(f'Tile {tile_idx:04d}', fontsize=8)

        # ── Col 1: NIR False Colour ───────────────────────────────
        nir = image_np[:, :, [3, 2, 1]].astype(np.float32)
        nir = np.clip((nir - 200) / (3500 - 200), 0, 1)
        axes[row, 1].imshow(nir)

        # ── Col 2: SAR VV ─────────────────────────────────────────
        # VV is channel index 7 (after 7 optical bands)
        vv = image_np[:, :, 7].astype(np.float32)
        axes[row, 2].imshow(vv, cmap='gray', vmin=-25, vmax=-5)

        # ── Col 3: 4-Class Label ──────────────────────────────────
        im = axes[row, 3].imshow(mask_vals, cmap=cmap, norm=norm)

        # Class distribution as text on mask panel
        total = mask_vals.size
        dist  = ' | '.join([
            f'{c}:{100*np.sum(mask_vals==c)/total:.0f}%'
            for c in range(4)
        ])
        axes[row, 3].set_xlabel(dist, fontsize=7)

        # Remove axes ticks for all panels
        for col in range(n_cols):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])

    # ── Shared colorbar for mask column ──────────────────────────────
    cbar = fig.colorbar(
        im, ax=axes[:, 3],
        ticks=[0.5, 1.5, 2.5, 3.5],
        fraction=0.03, pad=0.04
    )
    cbar.ax.set_yticklabels(
        ['Background', 'Active Channel', 'Active Bar', 'Vegetated Bar'],
        fontsize=8
    )

    plt.suptitle(
        f'QA Sample Tiles — Training Reach\n'
        f'Window: {TRAIN_START} to {TRAIN_END}  |  '
        f'Showing {len(qa_data)} random tiles',
        fontsize=11, fontweight='bold', y=1.01
    )
    plt.tight_layout()

    fig.savefig(
        os.path.join(FIGURES, 'QA_sample_tiles.png'),
        dpi=150, bbox_inches='tight'
    )
    plt.show()
    print(f'\nQA figure saved → {FIGURES}/QA_sample_tiles.png')

    # ── Aggregate class distribution across all QA tiles ─────────────
    all_masks = np.concatenate([m.flatten() for _, _, m in qa_data])
    total     = all_masks.size
    print('\nAggregate class distribution across QA tiles:')
    print('-'*50)
    for c, name in names.items():
        n = np.sum(all_masks == c)
        print(f'  Class {c} {name:16s}: {n:>8,} px  ({100*n/total:.1f}%)')

    return qa_data

qa_sample_tiles(n_samples=10)

In [ ]:
import shutil
import os

# Clear old training tiles, metadata and splits
for folder in [
    os.path.join(TILES, 'training'),
    META,
    f'{SPLITS}/train',
    f'{SPLITS}/val',
    f'{SPLITS}/test'
]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

print('Old tiles, metadata and splits cleared.')
print('Now rerun run_full_extraction()')

In [ ]:
def run_full_extraction():
    print('='*60)
    print('FULL TRAINING TILE EXTRACTION')
    print('='*60)
    print('All tiles extracted regardless of braidplain content.')
    print('Background-dominant tiles are kept — population controlled')
    print('at split stage via stratified sampling (35% background).')
    print('='*60)

    print('\nBuilding 4-class labels for training reach...')
    layers = build_label(TRAINING_AOI, TRAIN_START, TRAIN_END)
    print('Labels built.')

    print('Generating tile grid...')
    tiles = generate_tile_grid(TRAINING_AOI, TILE_SIZE_M, OVERLAP_M, CRS)

    stats = {'extracted': 0, 'skipped': 0, 'failed': 0}
    log   = []

    for idx, tile_geom in enumerate(tqdm(tiles, desc='Extracting tiles')):
        tile_id         = f'train_tile_{idx:04d}'
        success, status = extract_tile(
            layers, tile_geom, tile_id, 'training'
        )

        if not success:
            stats['failed'] += 1
            log.append({'tile': tile_id, 'status': status})
        elif status == 'skipped':
            stats['skipped'] += 1
        else:
            stats['extracted'] += 1

        # Progress update every 100 tiles
        if (idx + 1) % 100 == 0:
            print(f'  [{idx+1}/{len(tiles)}] '
                  f'extracted={stats["extracted"]} '
                  f'skipped={stats["skipped"]} '
                  f'failed={stats["failed"]}')

    print('\n── Extraction Summary ──────────────────────')
    for k, v in stats.items():
        print(f'  {k:12s}: {v}')
    print(f'  Total processed: {sum(stats.values())}')

    # Breakdown by tile type using saved metadata
    bp_count = 0
    bg_count = 0
    meta_files = os.listdir(META)
    for mf in meta_files:
        if not mf.startswith('train_tile'):
            continue
        with open(os.path.join(META, mf)) as f:
            meta = json.load(f)
        if meta.get('tile_type') == 'background':
            bg_count += 1
        else:
            bp_count += 1

    total = bp_count + bg_count
    if total > 0:
        print(f'\n── Tile Type Breakdown ─────────────────────')
        print(f'  Braidplain tiles : {bp_count} ({100*bp_count/total:.1f}%)')
        print(f'  Background tiles : {bg_count} ({100*bg_count/total:.1f}%)')
        print(f'\n  NOTE: Background will be subsampled to ~35% at split stage.')
        print(f'  Run create_splits(target_bg_fraction=0.35) next.')

    # Save failure log
    if log:
        with open(os.path.join(LOGS, 'extraction_failures.json'), 'w') as f:
            json.dump(log, f, indent=2)
        print(f'\n  {len(log)} failures logged → {LOGS}/extraction_failures.json')

    return stats

# Run full extraction
stats = run_full_extraction()

In [ ]:
def create_splits(target_bg_fraction=0.35, train_frac=0.80,
                  val_frac=0.10, seed=42):
    """
    Stratified train/val/test split preserving background representation.

    target_bg_fraction : proportion of final dataset that should be
                         background-dominant tiles (recommended: 0.30-0.40)
    """
    import random
    random.seed(seed)

    tile_root        = os.path.join(TILES, 'training')
    braidplain_tiles = []
    background_tiles = []

    # ── Separate tiles by type using saved metadata ──────────────
    for tile_id in os.listdir(tile_root):
        mask_path = os.path.join(tile_root, tile_id, 'mask.npy')
        meta_path = os.path.join(META, f'{tile_id}.json')
        if not os.path.exists(mask_path):
            continue
        # Determine type from metadata if available, else recompute
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                meta = json.load(f)
            tile_type = meta.get('tile_type', 'braidplain')
        else:
            mask_vals = np.load(mask_path)[:, :, 0]
            tile_type = ('braidplain'
                         if np.sum(mask_vals > 0) / mask_vals.size >= 0.10
                         else 'background')

        if tile_type == 'background':
            background_tiles.append(tile_id)
        else:
            braidplain_tiles.append(tile_id)

    print(f'  Braidplain tiles available : {len(braidplain_tiles)}')
    print(f'  Background tiles available : {len(background_tiles)}')

    # ── Subsample background to target fraction ───────────────────
    # Avoids overwhelming the dataset with background
    # but ensures the model sees adequate background examples
    n_total_if_all = len(braidplain_tiles) + len(background_tiles)
    n_bp            = len(braidplain_tiles)

    # How many background tiles needed to hit target_bg_fraction?
    # n_bg / (n_bp + n_bg) = target_bg_fraction
    # n_bg = n_bp * target_bg_fraction / (1 - target_bg_fraction)
    n_bg_target = int(n_bp * target_bg_fraction / (1 - target_bg_fraction))
    n_bg_use    = min(n_bg_target, len(background_tiles))

    random.shuffle(background_tiles)
    selected_bg = background_tiles[:n_bg_use]

    print(f'  Background tiles selected  : {n_bg_use} '
          f'({100*n_bg_use/(n_bp+n_bg_use):.1f}% of dataset)')

    # ── Combine and shuffle ───────────────────────────────────────
    all_tiles = braidplain_tiles + selected_bg
    random.shuffle(all_tiles)
    n       = len(all_tiles)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)

    splits = {
        'train': all_tiles[:n_train],
        'val'  : all_tiles[n_train:n_train + n_val],
        'test' : all_tiles[n_train + n_val:]
    }

    for split_name, tile_list in splits.items():
        with open(os.path.join(SPLITS, f'{split_name}_tiles.json'), 'w') as f:
            json.dump(tile_list, f, indent=2)
        print(f'  {split_name:5s}: {len(tile_list)} tiles')

    print(f'\n  Total dataset : {n} tiles')
    print(f'  Braidplain    : {n_bp} ({100*n_bp/n:.1f}%)')
    print(f'  Background    : {n_bg_use} ({100*n_bg_use/n:.1f}%)')
    print(f'  Split         : {train_frac:.0%} / {val_frac:.0%} / '
          f'{1-train_frac-val_frac:.0%}')

    return splits

# Uncomment to run after full extraction:
splits = create_splits(target_bg_fraction=0.35)

In [ ]:
import concurrent.futures

def run_full_extraction_2025():
    print('='*60)
    print('FAST THREADED TILE EXTRACTION — 2025')
    print('='*60)

    # 2025 composite — same AOI, different date window
    layers = build_label(TRAINING_AOI, TRAIN_START_2, TRAIN_END_2)
    tiles  = generate_tile_grid(TRAINING_AOI, TILE_SIZE_M, OVERLAP_M, CRS)

    print(f'Generated {len(tiles)} candidate tiles')

    def extract_tile_worker_2025(args):
        idx, tile_geom, layers = args
        # Different prefix — prevents collision with 2024 tiles
        tile_id = f'train2025_tile_{idx:04d}'
        success, status = extract_tile(
            layers, tile_geom, tile_id, 'training_2025'  # different subfolder
        )
        return idx, success, status

    args_list = [(idx, tile_geom, layers)
                 for idx, tile_geom in enumerate(tiles)]

    stats = {'extracted': 0, 'skipped': 0, 'failed': 0}
    log   = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(extract_tile_worker_2025, args): args[0]
            for args in args_list
        }
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=len(futures),
            desc='Extracting 2025 tiles'
        ):
            idx, success, status = future.result()
            tile_id = f'train2025_tile_{idx:04d}'
            if not success:
                stats['failed'] += 1
                log.append({'tile': tile_id, 'status': status})
            elif status == 'skipped':
                stats['skipped'] += 1
            else:
                stats['extracted'] += 1

    # Tile type breakdown
    bp_count = bg_count = 0
    training_2025_dir = os.path.join(TILES, 'training_2025')
    if os.path.exists(training_2025_dir):
        for tile_id in os.listdir(training_2025_dir):
            meta_path = os.path.join(META, f'{tile_id}.json')
            if not os.path.exists(meta_path):
                continue
            with open(meta_path) as f:
                meta = json.load(f)
            if meta.get('tile_type') == 'braidplain':
                bp_count += 1
            else:
                bg_count += 1

    total = bp_count + bg_count
    print('\n── 2025 Extraction Summary ─────────────────')
    for k, v in stats.items():
        print(f'  {k:12s}: {v}')
    if total > 0:
        print('\n── Tile Type Breakdown ─────────────────────')
        print(f'  Braidplain : {bp_count} ({100*bp_count/total:.1f}%)')
        print(f'  Background : {bg_count} ({100*bg_count/total:.1f}%)')
        print('\n── Combined Dataset Estimate ───────────────')
        bp_2024 = 450   # your confirmed 2024 count
        bg_sub  = int((bp_2024 + bp_count) * 0.35 / 0.65)
        total_est = bp_2024 + bp_count + bg_sub
        train_est = int(total_est * 0.80)
        print(f'  2024 braidplain  : {bp_2024}')
        print(f'  2025 braidplain  : {bp_count}')
        print(f'  Background (35%) : {bg_sub}')
        print(f'  Total dataset    : ~{total_est}')
        print(f'  Training split   : ~{train_est}')

    if log:
        with open(os.path.join(LOGS, 'extraction_failures_2025.json'), 'w') as f:
            json.dump(log, f, indent=2)

    return stats

# Run overnight
stats_2025 = run_full_extraction_2025()

In [ ]:
def merge_2025_into_training(splits_dir, tiles_dir_2025,
                              target_bg_fraction=0.35, seed=42):
    """
    Adds 2025 tiles to existing train split only.
    Val and test remain 2024-only — no temporal leakage.
    """
    import random
    random.seed(seed)

    # ── Load existing 2024 splits — val and test unchanged ────────────
    with open(os.path.join(splits_dir, 'train_tiles.json')) as f:
        train_2024 = json.load(f)
    with open(os.path.join(splits_dir, 'val_tiles.json')) as f:
        val_tiles  = json.load(f)   # untouched
    with open(os.path.join(splits_dir, 'test_tiles.json')) as f:
        test_tiles = json.load(f)   # untouched

    print(f'Existing 2024 splits:')
    print(f'  Train : {len(train_2024)}')
    print(f'  Val   : {len(val_tiles)}  ← unchanged')
    print(f'  Test  : {len(test_tiles)}  ← unchanged')

    # ── Load 2025 tiles and separate by type ──────────────────────────
    bp_2025, bg_2025 = [], []
    for tile_id in os.listdir(tiles_dir_2025):
        meta_path = os.path.join(META, f'{tile_id}.json')
        if not os.path.exists(meta_path):
            continue
        with open(meta_path) as f:
            meta = json.load(f)
        if meta.get('tile_type') == 'braidplain':
            bp_2025.append(tile_id)
        else:
            bg_2025.append(tile_id)

    print(f'\n2025 tiles available:')
    print(f'  Braidplain : {len(bp_2025)}')
    print(f'  Background : {len(bg_2025)}')

    # ── Subsample 2025 background to 35% ─────────────────────────────
    n_bg_2025 = int(len(bp_2025) * target_bg_fraction / (1 - target_bg_fraction))
    n_bg_2025 = min(len(bg_2025), n_bg_2025)
    bg_2025_sampled = random.sample(bg_2025, n_bg_2025)
    all_2025_train  = bp_2025 + bg_2025_sampled
    random.shuffle(all_2025_train)

    print(f'\n2025 tiles added to training:')
    print(f'  Braidplain : {len(bp_2025)}')
    print(f'  Background : {n_bg_2025} (35% subsample)')
    print(f'  Total 2025 : {len(all_2025_train)}')

    # ── Merge into training split ─────────────────────────────────────
    train_merged = train_2024 + all_2025_train
    random.shuffle(train_merged)

    # ── Save updated splits ───────────────────────────────────────────
    # Overwrite train only — val and test files untouched on disk
    with open(os.path.join(splits_dir, 'train_tiles.json'), 'w') as f:
        json.dump(train_merged, f, indent=2)

    # Val and test already correct on disk — no need to rewrite
    print(f'\n── Final Split Summary ─────────────────────')
    print(f'  Train : {len(train_merged)} '
          f'({len(train_2024)} from 2024 + {len(all_2025_train)} from 2025)')
    print(f'  Val   : {len(val_tiles)} (2024 only — unchanged)')
    print(f'  Test  : {len(test_tiles)} (2024 only — unchanged)')
    print(f'  Total : {len(train_merged)+len(val_tiles)+len(test_tiles)}')
    print()
    print('train_tiles.json updated.')
    print('val_tiles.json and test_tiles.json unchanged.')

    return train_merged


train_merged = merge_2025_into_training(
    splits_dir    = SPLITS,
    tiles_dir_2025= os.path.join(TILES, 'training_2025')
)

## 6. PART B — Export Full-Scene Inference Imagery (2019–2025)

For the inference reach (Tezpur–Guwahati), we export full-scene GeoTIFFs to Google Drive.
No tiling is done here — the model will handle tiling internally during inference (Script 3).
One 9-band GeoTIFF per year, lean season composite, same band order as training tiles.

**Export uses GEE batch tasks (asynchronous) — check Tasks tab after running.**

In [ ]:
# ── Coverage check BEFORE export ────────────────────────────────────────
# Verify S1 and S2 data availability for each year in the inference reach
# Especially important for 2022-2023 due to Sentinel-1B retirement (Aug 2022)

def check_coverage(year, start, end):
    aoi = INFERENCE_AOI

    s2_count = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(aoi)
                .filterDate(start, end)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD))
                .size().getInfo())

    s1_count = (ee.ImageCollection('COPERNICUS/S1_GRD')
                .filterBounds(aoi)
                .filterDate(start, end)
                .filter(ee.Filter.eq('instrumentMode', 'IW'))
                .filter(ee.Filter.eq('orbitProperties_pass', S1_PASS))
                .size().getInfo())

    flag = '⚠️ LOW' if (s1_count < 2 or s2_count < 2) else '✓ OK'
    print(f'  {year}: S2={s2_count:3d} scenes  S1={s1_count:3d} scenes  {flag}')
    return s2_count, s1_count

print('Coverage check for inference reach (Tezpur–Guwahati), lean seasons:')
print('-'*65)
for year, (start, end) in INFERENCE_YEARS.items():
    check_coverage(year, start, end)

In [ ]:
# ── Export full-scene inference composites to Google Drive ───────────────
# Exports one 9-band GeoTIFF per year to Drive folder BRAHMA_MORPHO/inference/scenes/
# File naming: inference_{year}_B2B3B4B8B8A_B11B12_VV_VH.tif
#
# NOTE: These are GEE batch exports — they run asynchronously.
#       Monitor progress in the GEE Code Editor Tasks tab.
#       Exports typically take 15-45 minutes per year depending on scene size.

def export_inference_scene(year, start, end, drive_folder='BRAHMA_MORPHO/inference/scenes'):
    """
    Export one annual lean-season composite for the inference reach to Drive.
    Band order matches training tiles: B2,B3,B4,B8,B8A,B11,B12,VV,VH
    """
    print(f'  Preparing export for {year} ({start} to {end})...')

    stacked, s2, s1 = build_s1s2_composite(INFERENCE_AOI, start, end)

    # Verify composite is not empty
    band_names = stacked.bandNames().getInfo()
    if not band_names:
        print(f'  WARNING: {year} composite is empty — skipping')
        return None

    desc    = f'inference_{year}'
    filename = f'inference_{year}_9band'

    task = ee.batch.Export.image.toDrive(
        image          = stacked.toFloat(),
        description    = desc,
        folder         = drive_folder,
        fileNamePrefix = filename,
        region         = INFERENCE_AOI,
        scale          = RESOLUTION,
        crs            = CRS,
        maxPixels      = 1e13,
        fileFormat     = 'GeoTIFF'
    )
    task.start()
    print(f'  ✓ Export task started: {desc} — monitor in GEE Tasks tab')
    return task


def export_all_inference_scenes():
    print('='*60)
    print('EXPORTING ANNUAL INFERENCE SCENES (2019-2025)')
    print('Reach: Tezpur to Guwahati')
    print('Bands: B2,B3,B4,B8,B8A,B11,B12,VV,VH (9 channels)')
    print('='*60)

    tasks = {}
    for year, (start, end) in INFERENCE_YEARS.items():
        task = export_inference_scene(year, start, end)
        if task:
            tasks[year] = task

    print(f'\n{len(tasks)} export tasks submitted.')
    print('Check GEE Tasks tab for progress.')
    print('Files will appear in Google Drive: BRAHMA_MORPHO/inference/scenes/')
    return tasks

# Run exports
tasks = export_all_inference_scenes()

In [ ]:
# ── Monitor export task status ───────────────────────────────────────────
# Run this cell to check status of all submitted export tasks

def check_task_status(tasks):
    print('Export task status:')
    print('-'*50)
    for year, task in tasks.items():
        status = task.status()
        state  = status.get('state', 'UNKNOWN')
        msg    = status.get('error_message', '')
        print(f'  {year}: {state}  {msg}')

# Uncomment to check:
# check_task_status(tasks)

In [ ]:

Map, layers = visualise_index_labels(
    INFERENCE_AOI,
    '2025-02-01',
    '2025-03-31',
    zoom   = 10,
    center = [26.4, 92.2]
)
Map

In [ ]:


def export_inference_validation_mask(drive_folder='BRAHMA_MORPHO/inference/scenes'):
    print('='*60)
    print('GENERATING INFERENCE REACH VALIDATION MASK — 2025')
    print('Reach: Tezpur to Guwahati')
    print('='*60)

    inf_start, inf_end = INFERENCE_YEARS['2025']

    print(f'\nBuilding 4-class labels for inference reach...')
    print(f'Window: {inf_start} to {inf_end}')



    layers = build_label(
        INFERENCE_AOI,
        inf_start,
        inf_end,

    )
    # ── Quick class distribution check ──────────────────────────────
    # Server-side pixel count to verify all four classes are present
    class_counts = layers['mask'].reduceRegion(
        reducer   = ee.Reducer.frequencyHistogram(),
        geometry  = INFERENCE_AOI,
        scale     = 30,          # coarser scale for speed
        bestEffort= True,
        maxPixels = 1e10
    ).getInfo()

    print('\nClass distribution (approx at 30m):')
    names = {'0':'Background', '1':'Active Channel',
             '2':'Active Bar', '3':'Vegetated Bar'}
    hist  = class_counts.get('mask', {})
    total = sum(hist.values()) if hist else 0
    for k, name in names.items():
        count = hist.get(k, 0)
        pct   = 100 * count / total if total > 0 else 0
        print(f'  Class {k} {name:16s}: {count:>10,.0f} px  ({pct:.1f}%)')

    if not hist:
        print('WARNING: mask appears empty — check inference AOI and braidplain coverage')
        return None

    # ── Export mask GeoTIFF to Drive ─────────────────────────────────
    task = ee.batch.Export.image.toDrive(
        image          = layers['mask'].toByte(),
        description    = 'inference_2025_validation_mask',
        folder         = drive_folder,
        fileNamePrefix = 'inference_2025_validation_mask',
        region         = INFERENCE_AOI,
        scale          = RESOLUTION,
        crs            = CRS,
        maxPixels      = 1e13,
        fileFormat     = 'GeoTIFF'
    )
    task.start()
    print(f'\n✓ Validation mask export started')
    print(f'  File: inference_2025_validation_mask.tif')
    print(f'  Folder: {drive_folder}')
    print(f'  Monitor in GEE Tasks tab')

    # ── Also export a quick visual QA of the inference reach label ───
    # Single-tile visual check from the inference reach centre
    print('\nExtracting one sample tile from inference reach for visual QA...')

    tiles  = generate_tile_grid(INFERENCE_AOI, TILE_SIZE_M, OVERLAP_M, CRS)
    centre = len(tiles) // 2

    braidplain_img = ee.Image(BRAIDPLAIN_ASSET).select('braidplain').eq(1)
    selected_geom  = None

    search_order = sorted(range(len(tiles)), key=lambda i: abs(i - centre))
    for idx in search_order[:30]:
        bp_sum = braidplain_img.reduceRegion(
            reducer=ee.Reducer.sum(), geometry=tiles[idx],
            scale=100, maxPixels=1e6
        ).getInfo()
        val = list(bp_sum.values())[0] if bp_sum else 0
        if val and val > 100:
            selected_geom = tiles[idx]
            print(f'  Selected tile index {idx}')
            break

    if selected_geom is None:
        print('  No valid tile found for visual QA')
        return task

    success, status = extract_tile(
        layers, selected_geom,
        'inference_2025_QA', 'inference_qa',
        force_reextract=True
    )

    if success:
        tile_dir = os.path.join(TILES, 'inference_qa', 'inference_2025_QA')
        image_np = np.load(os.path.join(tile_dir, 'image.npy'))
        mask_np  = np.load(os.path.join(tile_dir, 'mask.npy'))[:, :, 0]

        cmap = mcolors.ListedColormap(['#212121','#2196F3','#BDBDBD','#4CAF50'])
        norm = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle('Inference Reach (Tezpur-Guwahati) — 2025 Validation Label QA',
                     fontsize=12, fontweight='bold')

        # True colour
        rgb = image_np[:,:,[2,1,0]].astype(np.float32)
        rgb = np.clip((rgb - 200)/(2500 - 200), 0, 1)
        axes[0].imshow(rgb)
        axes[0].set_title('True Colour')
        axes[0].axis('off')

        # NIR false colour
        nir = image_np[:,:,[3,2,1]].astype(np.float32)
        nir = np.clip((nir - 200)/(3500 - 200), 0, 1)
        axes[1].imshow(nir)
        axes[1].set_title('NIR False Colour')
        axes[1].axis('off')

        # SAR VV
        axes[2].imshow(image_np[:,:,7], cmap='gray', vmin=-25, vmax=-5)
        axes[2].set_title('SAR VV')
        axes[2].axis('off')

        # Label mask
        im = axes[3].imshow(mask_np, cmap=cmap, norm=norm)
        axes[3].set_title('4-Class Label')
        axes[3].axis('off')
        cbar = plt.colorbar(im, ax=axes[3], ticks=[0.5,1.5,2.5,3.5])
        cbar.ax.set_yticklabels(
            ['Background','Channel','Active Bar','Veg. Bar'],
            fontsize=9
        )

        # Class distribution
        total = mask_np.size
        dist  = ' | '.join([f'{c}:{100*np.sum(mask_np==c)/total:.0f}%'
                            for c in range(4)])
        axes[3].set_xlabel(dist, fontsize=8)

        plt.tight_layout()
        fig.savefig(
            os.path.join(FIGURES, 'inference_2025_label_QA.png'),
            dpi=150, bbox_inches='tight'
        )
        plt.show()
        print(f'\nQA figure saved → {FIGURES}/inference_2025_label_QA.png')
    else:
        print(f'  QA tile extraction failed: {status}')

    return task

# Run validation mask generation
val_task = export_inference_validation_mask()


## 7. Label QA — Class Distribution Across All Extracted Tiles

In [ ]:
# ── Class distribution summary across all extracted training tiles ────────
# Run after full extraction to verify class balance
# If vegetated bar (class 3) is very rare, consider adjusting braidplain mask or
# applying class-weighted loss during training (handled in Script 2)

def summarise_class_distribution():
    tile_root = os.path.join(TILES, 'training')
    counts    = {0: 0, 1: 0, 2: 0, 3: 0}
    n_tiles   = 0

    for tile_id in os.listdir(tile_root):
        mask_path = os.path.join(tile_root, tile_id, 'mask.npy')
        if not os.path.exists(mask_path):
            continue
        mask_vals = np.load(mask_path)[:, :, 0]
        for c in range(4):
            counts[c] += int(np.sum(mask_vals == c))
        n_tiles += 1

    total     = sum(counts.values())
    names     = {0: 'Background', 1: 'Active Channel',
                 2: 'Active Bar',  3: 'Vegetated Bar'}

    print(f'Class distribution across {n_tiles} tiles:')
    print('-'*55)
    for c in range(4):
        pct = 100 * counts[c] / total if total > 0 else 0
        print(f'  Class {c} {names[c]:16s}: {counts[c]:>10,} px  ({pct:.1f}%)')
    print(f'  Total pixels: {total:,}')

    # Plot pie chart
    fig, ax = plt.subplots(figsize=(6, 6))
    colors  = ['#212121', '#2196F3', '#9E9E9E', '#4CAF50']
    labels  = [f'{names[c]}\n({100*counts[c]/total:.1f}%)' for c in range(4)]
    ax.pie([counts[c] for c in range(4)], labels=labels,
           colors=colors, startangle=90)
    ax.set_title('Training Dataset — Class Distribution')
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES, 'class_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved.')

    return counts

# Uncomment to run after full extraction:
# counts = summarise_class_distribution()

## 8. Output Summary

After running all cells, the following outputs are ready:

**Training data (for Script 2 — model training):**
```
BRAHMA_MORPHO/training/tiles/training/<tile_id>/
    image.npy   → shape (256, 256, 9)  float32  bands: B2,B3,B4,B8,B8A,B11,B12,VV,VH
    mask.npy    → shape (256, 256, 1)  uint8    values: 0,1,2,3
BRAHMA_MORPHO/training/splits/
    train_tiles.json  val_tiles.json  test_tiles.json
BRAHMA_MORPHO/training/metadata/<tile_id>.json
```

**Inference scenes (for Script 3 — multi-temporal inference):**
```
BRAHMA_MORPHO/inference/scenes/
    inference_2019_9band.tif   → GeoTIFF, 9 bands, ~10m, Tezpur-Guwahati reach
    inference_2020_9band.tif
    inference_2021_9band.tif
    inference_2022_9band.tif
    inference_2023_9band.tif
    inference_2024_9band.tif
    inference_2025_9band.tif
```

**Class definitions (consistent across both training and inference):**
- 0 = Background
- 1 = Active Channel
- 2 = Active Bar (bare sand/gravel)
- 3 = Vegetated Bar (spatial residual inside 28-yr braidplain mask — no NDVI)

**Next:** Run Script 2 (model training) with these tiles, then Script 3 (inference + morphometric analysis) with the GeoTIFFs.